In [1]:
import os

print("Working directory:", os.getcwd())
print("\nFiles here:")
for f in os.listdir('.'):
    print(' ', f)

Working directory: C:\Users\mithu

Files here:
  .ipynb_checkpoints
  .ipython
  .jupyter
  .matplotlib
  .ms-ad
  01_data_loading_and_sql.ipynb
  3D Objects
  AppData
  Apple
  Application Data
  canadian-grocery-analysis
  Contacts
  Cookies
  corporate_profits.csv
  cpi_categories.csv
  Cyntronex
  Documents
  Downloads
  events.csv
  Favorites
  food_prices.csv
  grocery_analysis.db
  iCloudDrive
  IntelGraphicsProfiles
  Links
  Local Settings
  Microsoft
  MicrosoftEdgeBackups
  Music
  My Documents
  NetHood
  NTUSER.DAT
  ntuser.dat.LOG1
  ntuser.dat.LOG2
  NTUSER.DAT{5daa6c9a-b4b9-11ef-a587-bd3093cee1d5}.TM.blf
  NTUSER.DAT{5daa6c9a-b4b9-11ef-a587-bd3093cee1d5}.TMContainer00000000000000000001.regtrans-ms
  NTUSER.DAT{5daa6c9a-b4b9-11ef-a587-bd3093cee1d5}.TMContainer00000000000000000002.regtrans-ms
  ntuser.ini
  OneDrive
  OneDrive - Cyntronex
  Pictures
  PrintHood
  Recent
  Saved Games
  Searches
  SendTo
  Start Menu
  Templates
  Videos


In [2]:
import os

# Files are already in the working directory — just set paths directly
CPI_FILE    = 'cpi_categories.csv'
FOOD_FILE   = 'food_prices.csv'
PROFIT_FILE = 'corporate_profits.csv'

# Verify
for f in [CPI_FILE, FOOD_FILE, PROFIT_FILE]:
    status = '✅ Found' if os.path.exists(f) else '❌ MISSING'
    print(f'{status}: {f}')

✅ Found: cpi_categories.csv
✅ Found: food_prices.csv
✅ Found: corporate_profits.csv


In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

cpi     = pd.read_csv('cpi_categories.csv', encoding='latin1')
food    = pd.read_csv('food_prices.csv', encoding='latin1')
profits = pd.read_csv('corporate_profits.csv')

print('CPI shape:',     cpi.shape)
print('Food shape:',    food.shape)
print('Profits shape:', profits.shape)

print('\n--- CPI columns ---');    print(cpi.columns.tolist())
print('\n--- Food columns ---');   print(food.columns.tolist())
print('\n--- Profits columns ---');print(profits.columns.tolist())

CPI shape: (15, 74)
Food shape: (110, 74)
Profits shape: (15, 5)

--- CPI columns ---
['ï»¿category', 'Feb-20', 'Mar-20', 'Apr-20', 'May-20', 'Jun-20', 'Jul-20', 'Aug-20', 'Sep-20', 'Oct-20', 'Nov-20', 'Dec-20', 'Jan-21', 'Feb-21', 'Mar-21', 'Apr-21', 'May-21', 'Jun-21', 'Jul-21', 'Aug-21', 'Sep-21', 'Oct-21', 'Nov-21', 'Dec-21', 'Jan-22', 'Feb-22', 'Mar-22', 'Apr-22', 'May-22', 'Jun-22', 'Jul-22', 'Aug-22', 'Sep-22', 'Oct-22', 'Nov-22', 'Dec-22', 'Jan-23', 'Feb-23', 'Mar-23', 'Apr-23', 'May-23', 'Jun-23', 'Jul-23', 'Aug-23', 'Sep-23', 'Oct-23', 'Nov-23', 'Dec-23', 'Jan-24', 'Feb-24', 'Mar-24', 'Apr-24', 'May-24', 'Jun-24', 'Jul-24', 'Aug-24', 'Sep-24', 'Oct-24', 'Nov-24', 'Dec-24', 'Jan-25', 'Feb-25', 'Mar-25', 'Apr-25', 'May-25', 'Jun-25', 'Jul-25', 'Aug-25', 'Sep-25', 'Oct-25', 'Nov-25', 'Dec-25', 'Jan-26', 'Feb-26']

--- Food columns ---
['ï»¿food_items', 'Feb-20', 'Mar-20', 'Apr-20', 'May-20', 'Jun-20', 'Jul-20', 'Aug-20', 'Sep-20', 'Oct-20', 'Nov-20', 'Dec-20', 'Jan-21', 'Feb-21'

In [4]:
# Rename first column
cpi.rename(columns={cpi.columns[0]: 'category'}, inplace=True)

# Remove footnote rows at bottom
cpi = cpi[cpi['category'].notna()]
cpi = cpi[~cpi['category'].str.contains('Foot|Source|Note|Table|Freq|Geo|Release', 
           na=False, case=False)]

# Melt from wide to long format
cpi_long = cpi.melt(id_vars=['category'], 
                     var_name='date', 
                     value_name='cpi_index')

# Clean date column
cpi_long['date'] = pd.to_datetime(cpi_long['date'], format='%b-%y')
cpi_long['cpi_index'] = pd.to_numeric(cpi_long['cpi_index'], errors='coerce')
cpi_long = cpi_long.dropna()
cpi_long = cpi_long.sort_values(['category', 'date']).reset_index(drop=True)

print("✅ CPI Long Format:")
print(cpi_long.shape)
print(cpi_long.head(5))

✅ CPI Long Format:
(1022, 3)
                                            category       date  cpi_index
0  Alcoholic beverages, tobacco products and recr... 2020-02-01      171.4
1  Alcoholic beverages, tobacco products and recr... 2020-03-01      171.5
2  Alcoholic beverages, tobacco products and recr... 2020-04-01      172.1
3  Alcoholic beverages, tobacco products and recr... 2020-05-01      172.1
4  Alcoholic beverages, tobacco products and recr... 2020-06-01      172.2


In [5]:
# Rename first column
food.rename(columns={food.columns[0]: 'food_item'}, inplace=True)

# Remove footnote rows
food = food[food['food_item'].notna()]
food = food[~food['food_item'].str.contains('Foot|Source|Note|Table|Freq|Geo|Release',
            na=False, case=False)]

# Melt from wide to long format
food_long = food.melt(id_vars=['food_item'],
                      var_name='date',
                      value_name='price_cad')

# Clean date column
food_long['date'] = pd.to_datetime(food_long['date'], format='%b-%y')
food_long['price_cad'] = pd.to_numeric(food_long['price_cad'], errors='coerce')
food_long = food_long.dropna()
food_long = food_long.sort_values(['food_item', 'date']).reset_index(drop=True)

print("✅ Food Prices Long Format:")
print(food_long.shape)
print(food_long.head(5))

✅ Food Prices Long Format:
(7884, 3)
              food_item       date  price_cad
0  Almonds, 200 grams 6 2020-02-01       5.13
1  Almonds, 200 grams 6 2020-03-01       5.34
2  Almonds, 200 grams 6 2020-04-01       5.50
3  Almonds, 200 grams 6 2020-05-01       5.52
4  Almonds, 200 grams 6 2020-06-01       5.57


In [6]:
# Calculate % change from Feb 2020 baseline for each food item
baseline = food_long[food_long['date'] == '2020-02-01'].copy()
baseline = baseline[['food_item', 'price_cad']].rename(
    columns={'price_cad': 'baseline_price'})

food_long = food_long.merge(baseline, on='food_item', how='left')
food_long['pct_change'] = ((food_long['price_cad'] - food_long['baseline_price']) 
                           / food_long['baseline_price'] * 100).round(2)

print("✅ Price Changes Calculated:")
print(food_long[['food_item', 'date', 'price_cad', 
                  'pct_change']].tail(10))

✅ Price Changes Calculated:
                food_item       date  price_cad  pct_change
7874  Yogurt, 500 grams 6 2025-05-01       3.70       42.86
7875  Yogurt, 500 grams 6 2025-06-01       3.67       41.70
7876  Yogurt, 500 grams 6 2025-07-01       3.70       42.86
7877  Yogurt, 500 grams 6 2025-08-01       3.69       42.47
7878  Yogurt, 500 grams 6 2025-09-01       3.67       41.70
7879  Yogurt, 500 grams 6 2025-10-01       3.72       43.63
7880  Yogurt, 500 grams 6 2025-11-01       3.73       44.02
7881  Yogurt, 500 grams 6 2025-12-01       3.74       44.40
7882  Yogurt, 500 grams 6 2026-01-01       3.78       45.95
7883  Yogurt, 500 grams 6 2026-02-01       3.80       46.72


In [9]:
import sqlite3

# Create database
conn = sqlite3.connect('grocery_analysis.db')

# Load all tables
cpi_long.to_sql('cpi_data', conn, if_exists='replace', index=False)
food_long.to_sql('food_prices', conn, if_exists='replace', index=False)
profits.to_sql('corporate_profits', conn, if_exists='replace', index=False)

print("✅ Database created: grocery_analysis.db")
print("\n📋 Tables:")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
for table in cursor.fetchall():
    cursor.execute(f"SELECT COUNT(*) FROM {table[0]}")
    print(f"   → {table[0]}: {cursor.fetchone()[0]} rows")

✅ Database created: grocery_analysis.db

📋 Tables:
   → cpi_data: 1022 rows
   → food_prices: 7884 rows
   → corporate_profits: 15 rows


In [10]:
def run_sql(query, title):
    print(f"\n{'='*55}")
    print(f"📌 {title}")
    print('='*55)
    result = pd.read_sql_query(query, conn)
    print(result.to_string(index=False))
    return result

# Query 1 — Most expensive food items in 2025
q1 = run_sql("""
    SELECT 
        food_item,
        ROUND(AVG(price_cad), 2) AS avg_price_2025
    FROM food_prices
    WHERE strftime('%Y', date) = '2025'
    GROUP BY food_item
    ORDER BY avg_price_2025 DESC
    LIMIT 10
""", "Top 10 Most Expensive Food Items in 2025")

# Query 2 — Biggest price increases since 2020
q2 = run_sql("""
    SELECT 
        food_item,
        ROUND(MIN(price_cad), 2) AS price_2020,
        ROUND(MAX(price_cad), 2) AS price_2025,
        ROUND(MAX(pct_change), 2) AS max_pct_increase
    FROM food_prices
    GROUP BY food_item
    ORDER BY max_pct_increase DESC
    LIMIT 10
""", "Top 10 Biggest Food Price Increases Since 2020")

# Query 3 — CPI Food vs Overall inflation
q3 = run_sql("""
    SELECT 
        strftime('%Y', date) AS year,
        ROUND(AVG(CASE WHEN category = 'Food 5' 
              THEN cpi_index END), 2) AS food_cpi,
        ROUND(AVG(CASE WHEN category = 'All-items' 
              THEN cpi_index END), 2) AS overall_cpi
    FROM cpi_data
    GROUP BY year
    ORDER BY year
""", "Food CPI vs Overall CPI by Year")

# Query 4 — Corporate profits during inflation
q4 = run_sql("""
    SELECT 
        company,
        year,
        revenue_billions_cad,
        net_profit_billions_cad,
        profit_margin_percent
    FROM corporate_profits
    ORDER BY company, year
""", "Corporate Profits During Inflation Crisis")

# Query 5 — Which year had worst food inflation
q5 = run_sql("""
    SELECT 
        strftime('%Y', date) AS year,
        ROUND(AVG(pct_change), 2) AS avg_price_increase_pct
    FROM food_prices
    GROUP BY year
    ORDER BY avg_price_increase_pct DESC
""", "Worst Years for Food Price Increases")


📌 Top 10 Most Expensive Food Items in 2025
                            food_item  avg_price_2025
         Infant formula, 900 grams  6           48.97
        Beef rib cuts, per kilogram 5           32.45
  Beef striploin cuts, per kilogram 5           30.90
               Salmon, per kilogram 5           26.41
Beef top sirloin cuts, per kilogram 5           25.95
    Beef stewing cuts, per kilogram 5           21.59
          Ground beef, per kilogram 5           14.69
      Chicken breasts, per kilogram 5           14.23
     Laundry detergent, 4.43 litres 6           13.42
                 Olive oil, 1 litre 6           12.65

📌 Top 10 Biggest Food Price Increases Since 2020
                            food_item  price_2020  price_2025  max_pct_increase
   Pork shoulder cuts, per kilogram 5        4.05        9.93            145.19
                 Olive oil, 1 litre 6        6.37       17.14            117.51
                   Cantaloupe, unit 5        2.18        4.65           

In [13]:
import os

# Create folder if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save all clean files
cpi_long.to_csv('data/cpi_clean.csv', index=False)
food_long.to_csv('data/food_prices_clean.csv', index=False)
profits.to_csv('data/corporate_profits_clean.csv', index=False)

# Save SQL query results
q1.to_csv('data/top_expensive_items.csv', index=False)
q2.to_csv('data/biggest_increases.csv', index=False)
q3.to_csv('data/food_vs_overall_cpi.csv', index=False)
q4.to_csv('data/corporate_profits_query.csv', index=False)   # fixed duplicate name
q5.to_csv('data/worst_years.csv', index=False)

conn.close()

print("✅ All files saved to C:\\Users\\mithu\\data\\")
print("\n🎉 Phase 1 Complete — Ready for Phase 2!")

✅ All files saved to C:\Users\mithu\data\

🎉 Phase 1 Complete — Ready for Phase 2!


---
## Phase 2 — Deep Analysis
### Basket Cost · Inflation Rates · Greedflation · Affordability · Volatility · Categories
---

In [14]:
# CELL A — Standard Grocery Basket Cost Over Time

basket_items = [
    'Ground beef, per kilogram 5',
    'Chicken breasts, per kilogram 5',
    'Pork shoulder cuts, per kilogram 5',
    'Eggs, 1 dozen 5',
    'Butter, 454 grams 5',
    'Milk, 2 litres 5',
    'Cheddar cheese, 500 grams 5',
    'Bread, 675 grams 5',
    'White sugar, 2 kilograms 6',
    'Canola oil, 3 litres 6',
    'Olive oil, 1 litre 6',
    'Pasta, 900 grams 6',
    'White rice, 2 kilograms 6',
    'Canned tomatoes, 796 millilitres 6',
    'Canned soup, 284 millilitres 6',
    'Potatoes, 4.54 kilograms 5',
    'Bananas, per kilogram 5',
    'Apples, per kilogram 5',
    'Strawberries, 454 grams 6',
    'Yogurt, 500 grams 6'
]

# Filter only basket items
basket = food_long[food_long['food_item'].isin(basket_items)].copy()

# Check which items matched
matched = basket['food_item'].unique()
print(f"✅ Matched {len(matched)} of {len(basket_items)} basket items\n")

missing = [i for i in basket_items if i not in matched]
if missing:
    print("❌ Not matched — check exact names in your data:")
    for m in missing:
        print(f"   {m}")

✅ Matched 17 of 20 basket items

❌ Not matched — check exact names in your data:
   Cheddar cheese, 500 grams 5
   Bread, 675 grams 5
   Pasta, 900 grams 6


In [15]:
# Run this to see exact item names in your data — use to fix mismatches above
print("All food items in your dataset:")
for item in sorted(food_long['food_item'].unique()):
    print(f"  '{item}'")

All food items in your dataset:
  'Almonds, 200 grams 6'
  'Apple juice, 2 litres 6'
  'Apples, per kilogram 5'
  'Avocado, unit 5'
  'Baby food, 128 millilitres 6'
  'Bacon, 500 grams 6'
  'Bananas, per kilogram 5'
  'Beef rib cuts, per kilogram 5'
  'Beef stewing cuts, per kilogram 5'
  'Beef striploin cuts, per kilogram 5'
  'Beef top sirloin cuts, per kilogram 5'
  'Block cheese, 500 grams 6'
  'Broccoli, unit 5'
  'Brown rice, 900 grams  6'
  'Butter, 454 grams 5'
  'Cabbage, per kilogram 5'
  'Canned baked beans, 398 millilitres 6'
  'Canned beans and lentils, 540 millilitres 5'
  'Canned corn, 341 millilitres 6'
  'Canned peach, 398 millilitres 6'
  'Canned pear, 398 millilitres 6'
  'Canned salmon, 213 grams 6'
  'Canned soup, 284 millilitres 6'
  'Canned tomatoes, 796 millilitres 6'
  'Canned tuna, 170 grams 5'
  'Canola oil, 3 litres 6'
  'Cantaloupe, unit 5'
  'Carrots, 1.36 kilograms 6'
  'Celery, unit 5'
  'Cereal, 400 grams 6'
  'Chicken breasts, per kilogram 5'
  'Chicke

In [16]:
basket_items = [
    'Ground beef, per kilogram 5',
    'Chicken breasts, per kilogram 5',
    'Pork shoulder cuts, per kilogram 5',
    'Eggs, 1 dozen 5',
    'Butter, 454 grams 5',
    'Milk, 2 litres 5',
    'Block cheese, 500 grams 6',          # was Cheddar cheese
    'White bread, 675 grams 6',           # was Bread
    'White sugar, 2 kilograms 6',
    'Canola oil, 3 litres 6',
    'Olive oil, 1 litre 6',
    'Dry or fresh pasta, 500 grams 6',    # was Pasta, 900 grams
    'White rice, 2 kilograms 6',
    'Canned tomatoes, 796 millilitres 6',
    'Canned soup, 284 millilitres 6',
    'Potatoes, 4.54 kilograms 5',
    'Bananas, per kilogram 5',
    'Apples, per kilogram 5',
    'Strawberries, 454 grams 6',
    'Yogurt, 500 grams 6'
]

basket = food_long[food_long['food_item'].isin(basket_items)].copy()
matched = basket['food_item'].unique()
print(f"✅ Matched {len(matched)} of {len(basket_items)} basket items")

✅ Matched 20 of 20 basket items


In [17]:
# CELL B — Monthly basket total cost 2020→2026

basket_monthly = (
    basket.groupby('date')['price_cad']
    .sum()
    .reset_index()
    .rename(columns={'price_cad': 'basket_cost'})
)

baseline_cost = basket_monthly[basket_monthly['date'] == '2020-02-01']['basket_cost'].values[0]
basket_monthly['basket_pct_change'] = ((basket_monthly['basket_cost'] - baseline_cost) / baseline_cost * 100).round(2)

print("✅ Basket Cost Over Time:")
print(f"   Feb 2020 baseline:  ${baseline_cost:.2f}")
print(f"   Feb 2026 cost:      ${basket_monthly[basket_monthly['date'] == '2026-02-01']['basket_cost'].values[0]:.2f}")
print(f"   Total increase:     {basket_monthly[basket_monthly['date'] == '2026-02-01']['basket_pct_change'].values[0]:.1f}%")
print(f"\n   Peak cost month:")
peak = basket_monthly.loc[basket_monthly['basket_cost'].idxmax()]
print(f"   {peak['date'].strftime('%b %Y')} — ${peak['basket_cost']:.2f} (+{peak['basket_pct_change']:.1f}%)")

✅ Basket Cost Over Time:
   Feb 2020 baseline:  $93.02
   Feb 2026 cost:      $126.49
   Total increase:     36.0%

   Peak cost month:
   Aug 2025 — $129.63 (+39.4%)


In [18]:
# CELL C — Category grouping + YoY inflation rates

category_map = {
    'Meat':    ['Ground beef, per kilogram 5', 'Chicken breasts, per kilogram 5',
                'Pork shoulder cuts, per kilogram 5'],
    'Dairy':   ['Eggs, 1 dozen 5', 'Butter, 454 grams 5', 'Milk, 2 litres 5',
                'Block cheese, 500 grams 6', 'Yogurt, 500 grams 6'],
    'Pantry':  ['White sugar, 2 kilograms 6', 'Canola oil, 3 litres 6',
                'Olive oil, 1 litre 6', 'Dry or fresh pasta, 500 grams 6',
                'White rice, 2 kilograms 6', 'Canned tomatoes, 796 millilitres 6',
                'Canned soup, 284 millilitres 6'],
    'Produce': ['Potatoes, 4.54 kilograms 5', 'Bananas, per kilogram 5',
                'Apples, per kilogram 5', 'Strawberries, 454 grams 6']
}

# Add category column
def get_category(item):
    for cat, items in category_map.items():
        if item in items:
            return cat
    return 'Other'

basket['category'] = basket['food_item'].apply(get_category)

# YoY average price by category
basket['year'] = basket['date'].dt.year
cat_yearly = (
    basket.groupby(['category', 'year'])['price_cad']
    .mean()
    .reset_index()
)

cat_yearly['yoy_change'] = cat_yearly.groupby('category')['price_cad'].pct_change() * 100
cat_yearly['yoy_change'] = cat_yearly['yoy_change'].round(2)

print("✅ Category Year-over-Year Inflation:")
print(cat_yearly[cat_yearly['year'] >= 2021].to_string(index=False))

✅ Category Year-over-Year Inflation:
category  year  price_cad  yoy_change
   Dairy  2021   4.445333        4.41
   Dairy  2022   4.830333        8.66
   Dairy  2023   5.112000        5.83
   Dairy  2024   5.177333        1.28
   Dairy  2025   5.297500        2.32
   Dairy  2026   5.370000        1.37
    Meat  2021   9.513889        2.54
    Meat  2022  10.504722       10.41
    Meat  2023  10.874722        3.52
    Meat  2024  11.209444        3.08
    Meat  2025  12.399722       10.62
    Meat  2026  12.311667       -0.71
   Other  2021   3.005000        1.02
   Other  2022   3.403333       13.26
   Other  2023   3.588333        5.44
   Other  2024   3.438333       -4.18
   Other  2025   3.487500        1.43
   Other  2026   3.605000        3.37
  Pantry  2021   4.607857        6.96
  Pantry  2022   5.532976       20.08
  Pantry  2023   6.252976       13.01
  Pantry  2024   6.420714        2.68
  Pantry  2025   6.020595       -6.23
  Pantry  2026   5.940000       -1.34
 Produce  202

In [19]:
# CELL D — Corporate margin vs Food CPI correlation

import numpy as np

# Food CPI yearly average
cpi_food = cpi_long[cpi_long['category'].str.contains('Food', case=False, na=False)].copy()
cpi_food['year'] = cpi_food['date'].dt.year
cpi_yearly = cpi_food.groupby('year')['cpi_index'].mean().reset_index()
cpi_yearly.columns = ['year', 'food_cpi']

# Merge with Loblaws (largest grocer — most meaningful signal)
loblaws = profits[profits['company'] == 'Loblaws'][['year', 'profit_margin_percent']].copy()
merged = loblaws.merge(cpi_yearly, on='year', how='inner')

correlation = merged['profit_margin_percent'].corr(merged['food_cpi'])

print("✅ Greedflation Analysis — Loblaws Margin vs Food CPI")
print(merged.to_string(index=False))
print(f"\n   Pearson Correlation:  {correlation:.3f}")
if correlation > 0.8:
    print("   📌 Strong positive correlation — margins rose AS food prices rose")
elif correlation > 0.5:
    print("   📌 Moderate positive correlation")
else:
    print("   📌 Weak correlation")

# All 3 grocers combined profit
profits['year'] = profits['year'].astype(int)
total_profit = profits.groupby('year')['net_profit_billions_cad'].sum().reset_index()
total_profit.columns = ['year', 'combined_profit']
baseline_profit = total_profit[total_profit['year'] == 2020]['combined_profit'].values[0]
total_profit['profit_growth_pct'] = ((total_profit['combined_profit'] - baseline_profit) / baseline_profit * 100).round(1)

print("\n✅ Combined Big 3 Grocer Net Profit Growth vs 2020:")
print(total_profit.to_string(index=False))

✅ Greedflation Analysis — Loblaws Margin vs Food CPI
 year  profit_margin_percent   food_cpi
 2020                    3.4 142.900000
 2021                    3.5 146.316667
 2022                    3.5 156.662500
 2023                    3.7 165.829167
 2024                    4.2 170.208333

   Pearson Correlation:  0.848
   📌 Strong positive correlation — margins rose AS food prices rose

✅ Combined Big 3 Grocer Net Profit Growth vs 2020:
 year  combined_profit  profit_growth_pct
 2020             3.15                0.0
 2021             3.53               12.1
 2022             3.76               19.4
 2023             3.80               20.6
 2024             4.39               39.4


In [20]:
# CELL E — Affordability Index (basket cost vs wage growth)

# Statistics Canada avg weekly earnings growth (actual data)
# Source: StatsCan Table 14-10-0063-01
wage_data = {
    2020: 1050.50,
    2021: 1093.22,
    2022: 1133.80,
    2023: 1180.44,
    2024: 1224.91,
    2025: 1262.00,
    2026: 1262.00   # held flat — 2026 wage data not yet published
}

wages = pd.DataFrame(list(wage_data.items()), columns=['year', 'avg_weekly_wage'])
basket_monthly['year'] = basket_monthly['date'].dt.year
basket_yearly = basket_monthly.groupby('year')['basket_cost'].mean().reset_index()

afford = basket_yearly.merge(wages, on='year', how='left')
afford['monthly_wage_approx'] = (afford['avg_weekly_wage'] * 52 / 12).round(2)
afford['basket_pct_of_monthly_wage'] = (afford['basket_cost'] / afford['monthly_wage_approx'] * 100).round(2)

# Hours of work to buy basket (based on avg hourly ~$30.88 in 2024)
hourly_wages = {2020: 27.50, 2021: 28.39, 2022: 29.53, 2023: 30.10, 2024: 30.88, 2025: 31.50, 2026: 31.50}
afford['avg_hourly_wage'] = afford['year'].map(hourly_wages)
afford['hours_to_buy_basket'] = (afford['basket_cost'] / afford['avg_hourly_wage']).round(2)

print("✅ Affordability Index:")
print(afford[['year', 'basket_cost', 'monthly_wage_approx',
              'basket_pct_of_monthly_wage', 'hours_to_buy_basket']].to_string(index=False))

✅ Affordability Index:
 year  basket_cost  monthly_wage_approx  basket_pct_of_monthly_wage  hours_to_buy_basket
 2020    96.736364              4552.17                        2.13                 3.52
 2021   101.068333              4737.29                        2.13                 3.56
 2022   114.065833              4913.13                        2.32                 3.86
 2023   122.473333              5115.24                        2.39                 4.07
 2024   124.060000              5307.94                        2.34                 4.02
 2025   125.617500              5468.67                        2.30                 3.99
 2026   124.845000              5468.67                        2.28                 3.96


In [21]:
# CELL F — Price Volatility Ranking (which items were most unstable)

volatility = (
    food_long.groupby('food_item')['price_cad']
    .agg(
        std_dev='std',
        mean_price='mean',
        min_price='min',
        max_price='max'
    )
    .reset_index()
)

volatility['coefficient_of_variation'] = (volatility['std_dev'] / volatility['mean_price'] * 100).round(2)
volatility['price_range_pct'] = ((volatility['max_price'] - volatility['min_price']) / volatility['min_price'] * 100).round(2)
volatility = volatility.sort_values('coefficient_of_variation', ascending=False)

print("✅ Top 15 Most Volatile Food Items (2020–2026):")
print(volatility[['food_item', 'mean_price', 'std_dev',
                   'coefficient_of_variation', 'price_range_pct']].head(15).to_string(index=False))

# Save everything for Power BI
import os
os.makedirs('data', exist_ok=True)

basket_monthly.to_csv('data/basket_cost_monthly.csv', index=False)
cat_yearly.to_csv('data/category_inflation.csv', index=False)
merged.to_csv('data/greedflation_correlation.csv', index=False)
total_profit.to_csv('data/combined_grocer_profits.csv', index=False)
afford.to_csv('data/affordability_index.csv', index=False)
volatility.to_csv('data/item_volatility.csv', index=False)

print("\n✅ All Phase 2 files saved to data/")
print("🎉 Phase 2 Analysis Complete — Ready for Power BI!")

✅ Top 15 Most Volatile Food Items (2020–2026):
                                food_item  mean_price  std_dev  coefficient_of_variation  price_range_pct
                     Olive oil, 1 litre 6   10.896575 3.195021                     29.32           169.07
                Strawberries, 454 grams 6    4.261096 0.968107                     22.72           132.41
                  Romaine lettuce, unit 5    3.097945 0.693682                     22.39           168.00
       Pork shoulder cuts, per kilogram 5    6.849726 1.444858                     21.09           145.19
             Infant formula, 900 grams  6   37.626027 7.775373                     20.66            78.25
    Beef top sirloin cuts, per kilogram 5   20.354247 4.169094                     20.48           127.20
                   Margarine, 907 grams 6    6.321370 1.252424                     19.81            86.52
                  Iceberg lettuce, unit 5    2.800000 0.537034                     19.18           130.16

In [22]:
import os

files_needed = [
    'data/basket_cost_monthly.csv',
    'data/category_inflation.csv',
    'data/greedflation_correlation.csv',
    'data/combined_grocer_profits.csv',
    'data/affordability_index.csv',
    'data/item_volatility.csv',
    'data/food_prices_clean.csv',
    'data/cpi_clean.csv',
    'data/corporate_profits_clean.csv'
]

print("Power BI Source Files:")
all_good = True
for f in files_needed:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    status = '✅' if exists else '❌'
    print(f"  {status} {f}  ({size:,} bytes)")
    if not exists:
        all_good = False

print(f"\n{'✅ All files ready — open Power BI!' if all_good else '❌ Fix missing files before proceeding'}")

Power BI Source Files:
  ✅ data/basket_cost_monthly.csv  (2,256 bytes)
  ✅ data/category_inflation.csv  (1,204 bytes)
  ✅ data/greedflation_correlation.csv  (159 bytes)
  ✅ data/combined_grocer_profits.csv  (132 bytes)
  ✅ data/affordability_index.csv  (473 bytes)
  ✅ data/item_volatility.csv  (9,622 bytes)
  ✅ data/food_prices_clean.csv  (441,354 bytes)
  ✅ data/cpi_clean.csv  (42,000 bytes)
  ✅ data/corporate_profits_clean.csv  (483 bytes)

✅ All files ready — open Power BI!
